### SQL Effort replaced by python script using Spark library :Scheduled Reports Analysis/extract tables out of SQL

In [0]:
%sql


DROP TABLE IF EXISTS dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source;
CREATE TABLE IF NOT EXISTS dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source (
  Document_Id STRING,
  Document_CUID STRING,
  Document_name STRING,
  Full_path STRING,
  updated_by STRING,
  source_DB_connection STRING,
  sql_table STRING,
  table_Name STRING,
  schema_Name STRING
);

In [0]:
%sql

select "Total" as category, count(distinct cms.Document_Id) as cnt from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
union all
select "left" as category, count(distinct cms.Document_Id) as cnt from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
left anti join dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as t
on cms.Document_Id = t.Document_Id
union all
select "processed" as category, count(distinct t.Document_Id) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as t
union all 
select "n4j_processed" as category, count(distinct documentCUID) as cnt from  dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables
;


-- SELECT DISTINCT cms.Document_Id 
-- FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
-- LEFT ANTI JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source AS t
-- ON cms.Document_Id = t.Document_Id
-- where cms.SQL_Query not in ("Error retrieving Query Plan", "Error retrieving Query Plan", "Data Source Type excel not handled for SQL extraction")
-- ORDER BY cms.Document_Id ASC
-- LIMIT 500;
-- select distinct * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
-- where Document_Id in ("10503698")
-- ;
-- select distinct * from dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables
-- where documentCUID="AV1snIJnv0lGr5VPkZNe8hI" 

-- select count(*) from 
-- -- dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms 
-- dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source
-- -- where Document_Id ="182571"
-- ;
-- DESCRIBE DETAIL dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source;


-- OPTIMIZE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source;


-- ALTER TABLE dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source
-- SET TBLPROPERTIES (
--   'delta.enableDeletionVectors' = 'false'
-- )



In [0]:
%sql
-- -- checking if I miss any data that´s in Neo4j output

-- select t1.documentCUID, t2.Document_Id, count(distinct t1.tableName) as n4j_cnt, count(distinct t2.table_Name) as regex_cnt   from 
-- dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables
-- as t1
-- left join dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as t2
-- on upper(trim(t1.documentCUID))=upper(trim(t2.Document_CUID))
-- and upper(trim(t1.tableName))=upper(trim(t2.table_Name))
-- where t2.Document_CUID is not null
-- group by  t1.documentCUID, t2.Document_Id
-- having n4j_cnt>regex_cnt
-- order by count(distinct t1.tableName) desc
-- ;

-- checking if I have more data than Neo4j output

select t1.documentCUID, t2.Document_Id, count(distinct t1.tableName) as n4j_cnt, count(distinct t2.table_Name) as regex_cnt   from 
dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source as t2
left join 
dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables
as t1
on upper(trim(t1.documentCUID))=upper(trim(t2.Document_CUID))
and upper(trim(t1.tableName))=upper(trim(t2.table_Name))
where
t2.Document_CUID is not null
-- and t2.Document_Id="10077545"
group by  t1.documentCUID, t2.Document_Id
having regex_cnt>n4j_cnt
order by count(distinct t1.tableName) desc

select sum(regex_cnt) from _sqldf


select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source
as
WITH cte_names AS (
  SELECT LOWER(cte_name) AS cte_name
  FROM (
    SELECT EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query,
        '(?i)with\\s+([a-zA-Z0-9_]+)\\s+as\\s*\\('
      )
    ) AS cte_name
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  )
)

SELECT DISTINCT
  Document_Id,
  Document_CUID, 
  Document_name, 
  Full_path, 
  lastAuthor as updated_by,
  Connection_Name as source_DB_connection, 
  -- SQL_Query,
  table_name
FROM (
  SELECT 
    Document_Id, 
    Document_CUID,
    Document_name, 
    Full_path, 
    lastAuthor,
    Connection_Name, 
    SQL_Query,
    EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query, 
        '(?i)(?:from|join)\\s+([a-zA-Z0-9_.]+)'
      )
    ) as table_name
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
) as t
left ANTI join cte_names as c
on t.table_name = c.cte_name
-- where document_id="10077545"

In [0]:
%sql
create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source
as
WITH cte_names AS (
  SELECT LOWER(cte_name) AS cte_name
  FROM (
    SELECT EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query,
        '(?i)with\\s+([a-zA-Z0-9_]+)\\s+as\\s*\\('
      )
    ) AS cte_name
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  )
),
-- Step 1: Extract FROM clause content (stops at next major SQL keyword)
from_blocks AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    from_block
  FROM (
    SELECT 
      Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
      EXPLODE(
        REGEXP_EXTRACT_ALL(
          SQL_Query,
          '(?is)\\bFROM\\b\\s+([a-zA-Z0-9_.\\[\\],\\s]+?)(?=\\s*\\b(?:WHERE|GROUP|ORDER|HAVING|UNION|LIMIT|INNER|LEFT|RIGHT|FULL|CROSS|NATURAL|JOIN|ON|SET)\\b|\\s*[;)]|\\s*$)'
        )
      ) as from_block
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  )
),
-- Step 2: Extract table names from FROM blocks (first identifier after start or comma)
from_table_names AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    EXPLODE(
      REGEXP_EXTRACT_ALL(
        from_block,
        '(?:^|,)\\s*([a-zA-Z0-9_.\\[\\]]+)'
      )
    ) as table_name
  FROM from_blocks
),
-- Step 3: Extract JOIN targets separately (unambiguous)
join_table_names AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query,
        '(?i)\\bJOIN\\b\\s+([a-zA-Z0-9_.\\[\\]]+)'
      )
    ) as table_name
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
),
-- Combine FROM tables and JOIN tables
all_table_refs AS (
  SELECT * FROM from_table_names
  UNION ALL
  SELECT * FROM join_table_names
)

SELECT DISTINCT
  Document_Id,
  Document_CUID, 
  Document_name, 
  Full_path, 
  lastAuthor as updated_by,
  Connection_Name as source_DB_connection, 
  trim((case when table_name like "%.%" then element_at(split(table_name, '\\.'), -1) else table_name end)) as tableName, 
  trim((case when table_name like "%.%" then element_at(split(table_name, '\\.'), -2) else null end)) as schemaName 
FROM all_table_refs as t
LEFT ANTI JOIN cte_names as c
ON LOWER(t.table_name) = c.cte_name

In [0]:
%sql
-- select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as cms
-- left join _sqldf as a1
-- on cms.DataSource_CUID=a1.documentCUID
-- where a1.documentCUID is not null

select * from (select distinct Document_Id, Document_CUID from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
where SQL_Query = "Error retrieving Query Plan") as E1
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables as T1
on upper(trim(E1.Document_CUID))=upper(trim(t1.documentCUID))

select distinct Document_Id from _sqldf



In [0]:
%sql
    
WITH cte_names AS (
  SELECT LOWER(cte_name) AS cte_name
  FROM (
    SELECT EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query,
        '(?i)with\\s+([a-zA-Z0-9_]+)\\s+as\\s*\\('
      )
    ) AS cte_name
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
  )
),
-- Method 1: Simple FROM/JOIN extraction from ORIGINAL SQL (catches tables inside subqueries)
simple_tables AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    EXPLODE(
      REGEXP_EXTRACT_ALL(
        SQL_Query,
        '(?i)\\b(?:FROM|JOIN)\\b\\s+([a-zA-Z0-9_.\\[\\]]+)'
      )
    ) as table_name
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
),
-- Method 2: Strip parentheses (3 passes for nested subqueries), then extract comma-separated tables from FROM blocks
cleaned_sql AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(SQL_Query, '\\([^()]*\\)', ' '),
        '\\([^()]*\\)', ' '
      ),
      '\\([^()]*\\)', ' '
    ) as clean_sql
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
),
from_blocks AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    from_block
  FROM (
    SELECT 
      Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
      EXPLODE(
        REGEXP_EXTRACT_ALL(
          clean_sql,
          '(?is)\\bFROM\\b\\s+([a-zA-Z0-9_.\\[\\],\\s]+?)(?=\\s*\\b(?:WHERE|GROUP|ORDER|HAVING|UNION|LIMIT|INNER|LEFT|RIGHT|FULL|CROSS|NATURAL|JOIN|ON|SET)\\b|\\s*;|\\s*$)'
        )
      ) as from_block
    FROM cleaned_sql
  )
),
comma_tables AS (
  SELECT 
    Document_Id, Document_CUID, Document_name, Full_path, lastAuthor, Connection_Name,
    EXPLODE(
      REGEXP_EXTRACT_ALL(
        from_block,
        '(?:^|,)\\s*([a-zA-Z0-9_.\\[\\]]+)'
      )
    ) as table_name
  FROM from_blocks
),
-- Combine both methods
all_table_refs AS (
  SELECT * FROM simple_tables
  UNION ALL
  SELECT * FROM comma_tables
)

SELECT DISTINCT
  Document_Id,
  Document_CUID, 
  Document_name, 
  Full_path, 
  lastAuthor as updated_by,
  -- Connection_Name as source_DB_connection, 
  trim((case when table_name like "%.%" then element_at(split(table_name, '\\.'), -1) else table_name end)) as tableName, 
  trim((case when table_name like "%.%" then element_at(split(table_name, '\\.'), -2) else null end)) as schemaName 
FROM all_table_refs as t
LEFT ANTI JOIN cte_names as c
ON LOWER(t.table_name) = c.cte_name
WHERE document_cuid="AYPilNC08GJHlHMwuDa__dk"

In [0]:
%sql
select 
Document_Id, count(*) as cnt from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms as t1
-- where t1.Document_CUID ="AYPilNC08GJHlHMwuDa__dk"
group by Document_Id
having cnt=20
;
-- select distinct * from dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables
-- where documentCUID="AYPilNC08GJHlHMwuDa__dk" 

In [0]:
%sql
MERGE INTO dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS meta
USING (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Document_CUID,
    cms.Folder_Id,
    cms.Full_path,
    cms.Document_name, 
    cms.updated,
    cms.scheduled,
    cms.created, 
    cms.lastAuthor,
    true as WEBI_Found_on_Server,
    cms.Execution_Time_in_seconds as Execution_Time_Sec,
    cms.Number_of_Data_Providers,
    cms.Number_of_API_Calls,
    cms.API_Pause_Counts, 
    cms.Start_Time, 
    cms.End_Time
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms AS cms
) AS src
ON meta.Document_Id = src.Document_Id
WHEN MATCHED THEN UPDATE SET
  meta.Document_CUID = src.Document_CUID,
  meta.Folder_Id = src.Folder_Id,
  meta.Full_path = src.Full_path,
  meta.Document_name = src.Document_name,
  meta.updated = src.updated,
  meta.scheduled = src.scheduled,
  meta.created = src.created,
  meta.lastAuthor = src.lastAuthor,
  meta.WEBI_Found_on_Server = src.WEBI_Found_on_Server,
  meta.Execution_Time_Sec = src.Execution_Time_Sec,
  meta.Number_of_Data_Providers = src.Number_of_Data_Providers,
  meta.Number_of_API_Calls = src.Number_of_API_Calls,
  meta.API_Pause_Counts = src.API_Pause_Counts,
  meta.Start_Time = src.Start_Time,
  meta.End_Time = src.End_Time
WHEN NOT MATCHED THEN INSERT (
  Document_Id,
  Document_CUID,
  Folder_Id,
  Full_path,
  Document_name,
  updated,
  scheduled,
  created,
  lastAuthor,
  WEBI_Found_on_Server,
  Execution_Time_Sec,
  Number_of_Data_Providers,
  Number_of_API_Calls,
  API_Pause_Counts,
  Start_Time,
  End_Time
) VALUES (
  src.Document_Id,
  src.Document_CUID,
  src.Folder_Id,
  src.Full_path,
  src.Document_name,
  src.updated,
  src.scheduled,
  src.created,
  src.lastAuthor,
  src.WEBI_Found_on_Server,
  src.Execution_Time_Sec,
  src.Number_of_Data_Providers,
  src.Number_of_API_Calls,
  src.API_Pause_Counts,
  src.Start_Time,
  src.End_Time
);

In [0]:
%sql 
select count(distinct Document_Id)  from dataplatform01_central_dev_catalog_01.custom_sap_bo.active_webi_source

